<a href="https://colab.research.google.com/github/pop123-ux/Qwen2.5-1.5B-Instruct_finetuned_for_seedance2.0_prompting/blob/main/cod.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install trl
!pip3 install deepspeed
!pip install --upgrade torchao
!pip install -U bitsandbytes>=0.46.1

In [ ]:
import pandas as pd
import requests
import io

# Updated URL to directly resolve the raw file content, including Git LFS files
url = "https://huggingface.co/datasets/GokuScraper/seedance-2-prompts-datasets/resolve/main/metadata.jsonl"

response = requests.get(url)
response.raise_for_status() # Raise an exception for HTTP errors
content = response.text # Get the content as a string

df = pd.read_json(io.StringIO(content), lines=True)

print(f"✅ Loaded {len(df)} structured video prompts!")

✅ Loaded 8484 structured video prompts!


In [5]:
!hf auth login

? How would you like to log in?  [Use arrows, Enter to confirm]
> Log in with your browser
  Paste an access token
? How would you like to log in? Log in with your browser

    Open this URL in your browser:
        https://hf.co/oauth/device

    And enter the code: D161-FALV

    Waiting for authorization...................
Token is valid.
The token `oauth-pop123ux` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `oauth-pop123ux`
Note: This token will be refreshed automatically when it expires.


In [14]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
import json
import os
import torch
from datasets import Dataset
from huggingface_hub import hf_hub_download
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 1. MONTARE DRIVE ȘI DIRECTORUL DE CHECKPOINT-URI
from google.colab import drive
drive.mount('/content/drive')

output_dir = "/content/drive/MyDrive/qwen-seedance-colab"

print("--- Pasul 1: Descărcare și procesare metadate ---")
repo_id = "GokuScraper/seedance-2-prompts-datasets"
metadata_path = hf_hub_download(repo_id=repo_id, filename="metadata.jsonl", repo_type="dataset")

processed_records = []
with open(metadata_path, 'r', encoding='utf-8') as f:
    for line in f:
        try:
            data = json.loads(line)
            prompt_en = data['i18n']['en']['p']

            category = data.get('category')
            if not category or category is None:
                category = "General"

            if isinstance(prompt_en, str) and prompt_en.strip():
                processed_records.append({"prompt": str(prompt_en), "category": str(category)})
        except:
            continue

print(f"Au fost încărcate cu succes {len(processed_records)} înregistrări valide.")

print("\n--- Pasul 2: Inițializare Tokenizer și Model format ChatML ---")
model_id = "Qwen/Qwen2-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Funcție manuală de tokenizare și mapare a etichetelor (Rezolvă eroarea inițială de Trainer)
def tokenize_and_map_labels(example):
    messages = [
        {"role": "system", "content": "You are an expert AI video prompt engineer. Generate highly detailed cinematic prompts for video generation models."},
        {"role": "user", "content": f"Generate a cinematic video prompt for the category: {example['category']}."},
        {"role": "assistant", "content": example['prompt']}
    ]
    # Aplicăm template-ul de chat nativ Qwen
    full_text = tokenizer.apply_chat_template(messages, tokenize=False)

    # Tokenizăm textul la lungimea fixă de 512
    tokenized = tokenizer(full_text, truncation=True, max_length=512, padding=False)

    # SOLUȚIA: Copiem input_ids direct în labels pentru ca modelul CausalLM să calculeze Loss-ul automat
    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

# Aplicăm procesarea manuală pe tot setul de date și ștergem coloanele text brute
raw_dataset = Dataset.from_list(processed_records)
tokenized_dataset = raw_dataset.map(
    tokenize_and_map_labels,
    remove_columns=raw_dataset.column_names
)

print("\n--- Pasul 3: Configurare bnb (4-bit QLoRA) optimizată pentru Tesla T4 ---")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.float16,
    trust_remote_code=True
)

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

print("\n--- Pasul 4: Configurare Argumente Trainer Clasic ---")
training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    max_steps=1200,
    learning_rate=2e-4,
    fp16=True,
    bf16=False,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    report_to="none"
)

# DataCollator adaugă padding-ul dinamic automat până la 512 unde este necesar
data_collator = DataCollatorForSeq2Seq(tokenizer, pad_to_multiple_of=8, return_tensors="pt", padding=True)

trainer = Trainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=training_args,
    data_collator=data_collator
)

print("\n--- Pasul 5: Pornire / Reluare Antrenare ---")
if os.path.exists(output_dir) and any("checkpoint" in f for f in os.listdir(output_dir)):
    print("S-au găsit salvări anterioare în Drive. Se reia antrenarea...")
    trainer.train(resume_from_checkpoint=True)
else:
    print("Se începe o sesiune de antrenare curată cu modelul stabil Qwen2...")
    trainer.train()

# Salvare finală a adaptorului LoRA direct în Drive
final_path = os.path.join(output_dir, "final_qwen_seedance_lora")
model.save_pretrained(final_path)
tokenizer.save_pretrained(final_path)
print(f"\nProces finalizat cu succes! Adaptorul LoRA a fost salvat în {final_path}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
--- Pasul 1: Descărcare și procesare metadate ---
Au fost încărcate cu succes 8483 înregistrări valide.

--- Pasul 2: Inițializare Tokenizer și Model format ChatML ---


Map:   0%|          | 0/8483 [00:00<?, ? examples/s]


--- Pasul 3: Configurare bnb (4-bit QLoRA) optimizată pentru Tesla T4 ---


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820

--- Pasul 4: Configurare Argumente Trainer Clasic ---

--- Pasul 5: Pornire / Reluare Antrenare ---
Se începe o sesiune de antrenare curată cu modelul stabil Qwen2...


Step,Training Loss
10,2.847360
20,2.601681
30,2.559332
40,2.515199
50,2.488214
60,2.498934
70,2.449146
80,2.389579
90,2.482402
100,2.448816


In [ ]:
df["duration"] = df["spec"].apply(lambda x: x.get("duration"))
df["ratio"] = df["spec"].apply(lambda x: x.get("ratio"))
df["width"] = df["spec"].apply(lambda x: x.get("width"))
df["height"] = df["spec"].apply(lambda x: x.get("height"))

In [ ]:
df['duration'].head()

,duration
0,15.10
1,15.13
2,15.12
3,15.10
4,15.13


In [ ]:
from datasets import Dataset

ds = Dataset.from_pandas(df)
ds

Dataset({
    features: ['category', 'date', 'file_name', 'i18n', 'id', 'is_featured', 'media', 'model_info', 'platform', 'raw_p', 'slug', 'sourceLink', 'spec', 'version'],
    num_rows: 8484
})

In [ ]:
ds.features

{'category': Value('string'),
 'date': Value('timestamp[ns]'),
 'file_name': Value('string'),
 'i18n': {'en': {'p': Value('string'),
   't': Value('string'),
   'tags': List(Value('string'))},
  'zh': {'p': Value('string'),
   't': Value('string'),
   'tags': List(Value('string'))}},
 'id': Value('string'),
 'is_featured': Value('bool'),
 'media': {'c': Value('string'),
  'ref_images': List(Value('string')),
  'v': Value('string')},
 'model_info': {'model': Value('string'),
  'name': Value('string'),
  'version': Value('string')},
 'platform': Value('string'),
 'raw_p': Value('string'),
 'slug': Value('string'),
 'sourceLink': Value('string'),
 'spec': {'duration': Value('float64'),
  'height': Value('int64'),
  'ratio': Value('float64'),
  'safety_rating': Value('string'),
  'width': Value('int64')},
 'version': Value('int64'),
 'duration': Value('float64'),
 'ratio': Value('float64'),
 'width': Value('int64'),
 'height': Value('int64')}

In [ ]:
ds['spec']['height']

Column([720, 720, 720, 720, 720])

In [ ]:
# Map the dataset to our needs

def convert(example):
    en = example["i18n"].get("en", {})
    spec = example["spec"]

    prompt = en.get("p") or example.get("raw_p") or ""

    return {
        "messages": [
            {
                "role": "user",
                "content": (
                    f"Write a Seedance 2 cinematic prompt.\n"
                    f"Category: {example['category']}\n"
                    f"Duration: {spec['duration']}\n"
                    f"Aspect Ratio: {spec['ratio']}"
                ),
            },
            {
                "role": "assistant",
                "content": prompt,
            },
        ]
    }

dataset = ds.map(convert)

Map:   0%|          | 0/8484 [00:00<?, ? examples/s]

In [ ]:
# The part of the dataset that contains empty columns cannot be properly parsed by the "tokenize" function we're about to write

for i, ex in enumerate(dataset):
    for msg in ex["messages"]:
        if msg["content"] is None:
            print(i)
            print(ex)
            break

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    'Qwen/Qwen2.5-1.5B-Instruct'
)

# The 'tokenize' function I was writing about before
def tokenize(example):
  text = tokenizer.apply_chat_template(
      example['messages'],
      tokenize=False,
      add_generation_prompt=False
  )

  return tokenizer(
      text,
      truncation=True,
      max_length=4096
  )

dataset_f = dataset.map(tokenize)

Map:   0%|          | 0/8484 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    device_map='auto',
    dtype='bfloat16'
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
# Add LoRA
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=64,
    lora_alpha=128,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset_f,
    args=SFTConfig(
        output_dir="./seedance-with-lora",
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.03,
        num_train_epochs=3,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        gradient_checkpointing=True,
        bf16=True,
        logging_steps=10,
        save_steps=500,
        packing=False,
        max_seq_length=1024,
    ),
)

model.config.use_cache = False

trainer.train()

Building labels for train dataset:   0%|          | 0/8484 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/8484 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/8484 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,2.895469
20,2.438393
30,2.410674
40,2.442296
50,2.388594
60,2.406348
70,2.344794
80,2.356938
90,2.305909
100,2.397007


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
model.push_to_hub('pop123-ux/seedance-qwen2.5-7b-lora')
tokenizer.push_to_hub('pop123-ux/seedance-qwen2.5-7b-lora')

In [ ]:
# Load model example
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-7B-Instruct")
model = PeftModel.from_pretrained(
    base,
    "your-username/seedance-qwen2.5-7b-lora"
)

tokenizer = AutoTokenizer.from_pretrained("your-username/seedance-qwen2.5-7b-lora")